# Final Corpus Classification — GPT-5.4-mini (v2)

Batch classification of the final corpus using a single API call per corpus type to minimise token usage.

> **Note:** The standard / irony / obfuscated splits share identical test sets within each corpus type. Classification is therefore run once per corpus type (raw-corpus and pre-filtered-corpus).

## Dependencies

In [ ]:
import os
import re

from openai import OpenAI

import pandas as pd

from sklearn.metrics import confusion_matrix, classification_report

import matplotlib.pyplot as plt
import seaborn as sns

import watermark

%load_ext watermark
%matplotlib inline

plt.style.use('../style.mplstyle')
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

In [ ]:
%watermark -n -v -m -iv

## OpenAI client

In [ ]:
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

## Batch classifier

Sends **all tweets in a single API call**. The response is a numbered list (`1. POSITIVE`, `2. NEGATIVE`, …) that is parsed back to a Series.

In [ ]:
SYSTEM_MSG = "Sos un clasificador binario de tweets que hablan sobre drogas ilícitas"

CRITERIA = """\
    Clasifica cada tweet como POSITIVE o NEGATIVE según estos criterios:

    POSITIVE: cumple con uno o más de los siguientes:
    - El usuario del tweet habla de cómo o qué tipo de droga ilícita está consumiendo.
    - El usuario del tweet expresa la necesidad de consumir drogas ilícitas, ya sea por abstinencia o por gusto.
    - El usuario añora consumir drogas ilícitas.

    NEGATIVE: no cumple con ningún criterio POSITIVE, por ejemplo:
    - Habla sobre noticias o información general sobre drogas ilícitas.
    - Menciona drogas ilícitas sin relación con consumo problemático o necesidad.
    - Expresa ironía o sarcasmo relacionado con drogas ilícitas.

    Tener en cuenta los siguientes aspectos:
    - En el tweet puede estar presente la ironía o sarcasmo.
    - El análisis se centra en el autor del tweet. Por ejemplo, si el tweet cita a otro usuario y le pregunta si ha consumido drogas, o si habla en nombre de otro usuario mencionando que ese usuario consume drogas, la clasificación del tweet debe ser NEGATIVE, ya que el contenido no involucra directamente al autor.
    - Algunos tweets mencionan tomar una línea de colectivo, subte o tren, pero solamente esto, no es condición suficiente para interpretarlo como una referencia al consumo de drogas ilícitas."""

def build_batch_prompt(tweets: list[str]) -> str:
    numbered = "\n".join(f"{i+1}. {t}" for i, t in enumerate(tweets))
    return (
        CRITERIA
        + "\n\n"
        + "Responde con una lista numerada, exactamente una clasificación por línea, "
        + "únicamente la palabra POSITIVE o NEGATIVE (sin puntuación adicional ni explicaciones):\n\n"
        + numbered
    )

def parse_batch_response(response_text: str, n: int) -> list[str]:
    lines = [l.strip() for l in response_text.strip().splitlines() if l.strip()]
    results = []
    for line in lines:
        # strip optional leading "N." or "N)"
        cleaned = re.sub(r"^\d+[.)\s]+", "", line).strip().upper().rstrip(".")
        if cleaned in ("POSITIVE", "NEGATIVE"):
            results.append(cleaned)
    if len(results) != n:
        raise ValueError(f"Expected {n} classifications, got {len(results)}. Response:\n{response_text}")
    return results

def classify_batch(tweets: list[str]) -> list[str]:
    prompt = build_batch_prompt(tweets)
    response = client.chat.completions.create(
        model="gpt-5.4-mini",
        messages=[
            {"role": "system", "content": SYSTEM_MSG},
            {"role": "user", "content": prompt},
        ],
        temperature=1,
    )
    return parse_batch_response(response.choices[0].message.content, len(tweets))

## Corpus types

Each entry is `(label, path_to_test_csv, output_prefix)`.

In [ ]:
CORPORA = [
    ("raw-corpus",           "../data/raw/final-corpus/raw-corpus/standard/test.csv",           f"raw-corpus_v2"),
    ("pre-filtered-corpus",  "../data/raw/final-corpus/pre-filtered-corpus/standard/test.csv",  f"pre-filtered-corpus_v2"),
]


## Run batch classification

In [ ]:
results = {}

for corpus_label, csv_path, out_prefix in CORPORA:
    print(f"\n{'='*60}")
    print(f"Corpus: {corpus_label}")
    print(f"{'='*60}")

    df = pd.read_csv(csv_path)
    tweets = df["text"].tolist()
    labels = df["label"].tolist()

    print(f"Tweets to classify: {len(tweets)}")

    predictions = classify_batch(tweets)
    df["prediction"] = predictions

    results[corpus_label] = df
    print(f"Done. Predictions: {df['prediction'].value_counts().to_dict()}")

## Metrics per corpus

In [ ]:
LABELS = ["POSITIVE", "NEGATIVE"]

for corpus_label, _, out_prefix in CORPORA:
    df = results[corpus_label]
    y_true = df["label"]
    y_pred = df["prediction"]

    print(f"\n{'='*60}")
    print(f"Corpus: {corpus_label}")
    print(f"{'='*60}")

    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred, labels=LABELS)
    fig, ax = plt.subplots(figsize=(2, 1))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=LABELS, yticklabels=LABELS,
                cbar=True, annot_kws={"size": 9}, ax=ax)
    ax.collections[0].colorbar.ax.tick_params(labelsize=6)
    ax.set_xlabel("Prediction", fontsize=6)
    ax.set_ylabel("True", fontsize=6)
    ax.tick_params(labelsize=5)
    ax.set_title(f"Confusion Matrix — {corpus_label}", fontsize=9)
    plt.tight_layout()
    plt.show()
    print(cm)

    # Classification report
    report = classification_report(y_true, y_pred, labels=LABELS, output_dict=True)
    report_df = pd.DataFrame(report).transpose()
    print("\nClassification report:")
    display(report_df.style.format({"precision": "{:.2f}", "recall": "{:.2f}", "f1-score": "{:.2f}", "support": "{:.0f}"}))

    # Errors
    df["error"] = df["label"] != df["prediction"]
    errors = df[df["error"]][["text", "label", "prediction"]]
    print(f"\nWrongly classified: {len(errors)}")
    display(errors)

## Save wrongly-classified tweets

In [ ]:
for corpus_label, _, out_prefix in CORPORA:
    df = results[corpus_label]
    errors = df[df["label"] != df["prediction"]][["text", "label", "prediction"]]
    out_path = f"../data/processed/final-corpus/{corpus_label}/wct_gpt-5.4-mini-v2.csv"
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    errors.to_csv(out_path, index=False)
    print(f"Saved {len(errors)} errors → {out_path}")